# Phase 7: Generate Training Dataset Using Validated Pinch Engine

This notebook generates a high-quality, physically validated dataset of thermal process stream networks and their exact Pinch Analysis targets ($Q_{H,\min}$, $Q_{C,\min}$, $T_{\text{pinch}}$).

Every case generated by the engine strictly obeys:
1. **Thermodynamic Feasibility**: $T_s > T_t$ for hot streams, $T_t > T_s$ for cold streams, $CP > 0$.
2. **First Law of Thermodynamics**: $\sum Q_{\text{hot}} + Q_{H,\min} = \sum Q_{\text{cold}} + Q_{C,\min}$.
3. **Second Law Pinch Principle**: Feasible cascade ($R_i \ge 0$) with exact pinch zero-point identification.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from pinchpy.ml.dataset import generate_dataset, generate_case
from pinchpy.ml.features import TARGET_NAMES

## 1. Generate Single Synthetic Case
Let's inspect a single generated network and its corresponding feature and target vectors.

In [ ]:
streams, x, y = generate_case(n_hot=2, n_cold=2, delta_t_min=10.0, seed=42)
for s in streams:
    print(f"{s.id} ({s.type}): {s.supply_temperature} -> {s.target_temperature} °C, CP={s.cp} kW/°C, Q={s.duty():.1f} kW")

print("\nTargets:")
print(f"  QH,min      = {y[0]:.2f} kW")
print(f"  QC,min      = {y[1]:.2f} kW")
print(f"  Pinch Temp* = {y[2]:.2f} °C")

## 2. Generate Full Training Dataset (1,000 cases)
We generate 1,000 diverse cases with varying numbers of hot and cold streams ($1$ to $5$) and discrete $\Delta T_{\min}$ approach values ($5, 10, 15, 20\,^\circ\text{C}$).

In [ ]:
output_file = Path("../data/raw/pinch_dataset_1000.csv")
df = generate_dataset(n_cases=1000, seed=42, max_hot=5, max_cold=5, output_path=output_file)
print(f"Dataset generated with shape: {df.shape}")
print(f"Saved to: {output_file.resolve()}")
df.head()

## 3. Verify Energy Balance on Generated Dataset
Let's verify that the First Law of Thermodynamics holds across all 1,000 generated cases.

In [ ]:
hot_duties = sum(df[f"H{i+1}_Q"] for i in range(5))
cold_duties = sum(df[f"C{i+1}_Q"] for i in range(5))
energy_imbalance = (hot_duties + df["minimum_hot_utility"]) - (cold_duties + df["minimum_cold_utility"])
max_err = energy_imbalance.abs().max()
print(f"Maximum energy balance discrepancy across 1,000 cases: {max_err:.2e} kW")
assert max_err < 1e-4, "Energy balance violated!"

## 4. Summary Statistics of Targets

In [ ]:
df[TARGET_NAMES].describe()